# Notebook 05: Deep Learning Neural Network Benchmarking (MLP, GRU, & LSTM)

In this notebook, we systematically evaluate Deep Learning sequence architectures for metro passenger forecasting.

**Input:** `data/metro_processed.csv`

**Deep Learning Benchmarking Strategy:**
1. **Baseline MLP:** Multi-Layer Perceptron (Dense Feed-Forward Network without recurrent memory)
2. **Baseline GRU:** Gated Recurrent Unit Network (Lighter single-gate RNN baseline)
3. **Tuned Bi-LSTM:** Stacked Bidirectional Long Short-Term Memory Network with Dropout regularization

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GRU, LSTM, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(42)
np.random.seed(42)
print(f'TensorFlow Version: {tf.__version__}')

TensorFlow Version: 2.20.0


In [2]:
df = pd.read_csv('data/metro_processed.csv')
df['DateTime'] = pd.to_datetime(df['DateTime'])
print(f'Dataset Shape: {df.shape}')
df.head()

Dataset Shape: (93624, 24)


,Date,Hour,Station,Boarding_Count,Exit_Count,DateTime,Boarding_Capped,DayOfWeek,DayOfMonth,WeekOfYear,...,Day_Cos,Is_Morning_Peak,Is_Evening_Peak,Is_Peak_Hour,Lag_1h,Lag_2h,Lag_24h,Rolling_3h,Rolling_3h_Std,Station_AvgTraffic
0,2025-08-02,0,Attiguppe,0,0,2025-08-02 00:00:00,0,5,2,31,...,-0.222521,0,0,0,23.0,141.0,0.0,122.000000,91.000000,350.94184
1,2025-08-02,1,Attiguppe,0,0,2025-08-02 01:00:00,0,5,2,31,...,-0.222521,0,0,0,0.0,23.0,0.0,54.666667,75.646106,350.94184
2,2025-08-02,2,Attiguppe,0,0,2025-08-02 02:00:00,0,5,2,31,...,-0.222521,0,0,0,0.0,0.0,0.0,7.666667,13.279056,350.94184
3,2025-08-02,3,Attiguppe,0,0,2025-08-02 03:00:00,0,5,2,31,...,-0.222521,0,0,0,0.0,0.0,0.0,0.000000,0.000000,350.94184
4,2025-08-02,4,Attiguppe,10,0,2025-08-02 04:00:00,10,5,2,31,...,-0.222521,0,0,0,0.0,0.0,6.0,0.000000,0.000000,350.94184


In [3]:
split_date = pd.to_datetime('2025-09-20')
train = df[df['DateTime'] < split_date].copy()
test = df[df['DateTime'] >= split_date].copy()
target = 'Boarding_Count'

## 1. Clean Data Partitioning & Station-Aware Sequence Windowing
We use 20 clean features (excluding `Boarding_Capped`) and slice sequence windows strictly **per station**.

In [5]:
features_clean = [
    'Hour', 'Hour_Sin', 'Hour_Cos',
    'DayOfWeek', 'Day_Sin', 'Day_Cos', 'DayOfMonth', 'WeekOfYear',
    'Is_Weekend', 'Is_Morning_Peak', 'Is_Evening_Peak', 'Is_Peak_Hour','Exit_Count', 'Lag_1h', 'Lag_2h', 'Lag_24h', 'Rolling_3h', 'Rolling_3h_Std', 'Station_AvgTraffic'
]

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

scaler_X.fit(train[features_clean])
scaler_y.fit(train[[target]])

def transform_df(data_df):
    s_df = data_df.copy()
    s_df[features_clean] = scaler_X.transform(data_df[features_clean])
    s_df[target] = scaler_y.transform(data_df[[target]])
    return s_df

train_scaled = transform_df(train)
test_scaled = transform_df(test)

def create_station_sequences(scaled_df, feature_cols, target_col, seq_length=24):
    X_seqs, y_seqs, meta_rows = [], [], []
    for station_name, group in scaled_df.groupby('Station'):
        group_sorted = group.sort_values('DateTime').reset_index(drop=True)
        X_vals = group_sorted[feature_cols].values
        y_vals = group_sorted[target_col].values
        for i in range(seq_length, len(group_sorted)):
            X_seqs.append(X_vals[i - seq_length:i])
            y_seqs.append(y_vals[i])
            meta_rows.append(group_sorted.iloc[i])
    return np.array(X_seqs), np.array(y_seqs), pd.DataFrame(meta_rows)

X_tr, y_tr, meta_tr = create_station_sequences(train_scaled, features_clean, target, seq_length=24)
X_te, y_te, meta_te = create_station_sequences(test_scaled, features_clean, target, seq_length=24)

print(f'Station-Aware Train Sequences: {X_tr.shape}')
print(f'Station-Aware Test Sequences:  {X_te.shape}')

Station-Aware Train Sequences: (69720, 24, 19)
Station-Aware Test Sequences:  (19920, 24, 19)


## 2. Deep Learning Baseline 1: Multi-Layer Perceptron (MLP)
Dense feed-forward baseline without recurrent memory layers.

In [6]:
X_tr_flat = X_tr.reshape(X_tr.shape[0], -1)
X_te_flat = X_te.reshape(X_te.shape[0], -1)

model_mlp = Sequential([
    Dense(128, activation='relu', input_shape=(X_tr_flat.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1)
])
model_mlp.compile(optimizer='adam', loss='mse')
model_mlp.fit(X_tr_flat, y_tr, epochs=8, batch_size=128, validation_split=0.15, verbose=1)

pred_mlp_s = model_mlp.predict(X_te_flat)
pred_mlp = np.clip(scaler_y.inverse_transform(pred_mlp_s).flatten(), 0, None)
y_true = scaler_y.inverse_transform(y_te.reshape(-1, 1)).flatten()

mlp_rmse = np.sqrt(mean_squared_error(y_true, pred_mlp))
mlp_mae = mean_absolute_error(y_true, pred_mlp)
mlp_r2 = r2_score(y_true, pred_mlp)
print(f'--- MLP Performance --- RMSE: {mlp_rmse:.2f}, MAE: {mlp_mae:.2f}, R²: {mlp_r2:.4f}')

c:\Users\tejas\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/8
463/463 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.0066 - val_loss: 0.0020
Epoch 2/8
463/463 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0015 - val_loss: 0.0015
Epoch 3/8
463/463 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0012 - val_loss: 0.0014
Epoch 4/8
463/463 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0010 - val_loss: 0.0013
Epoch 5/8
463/463 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 9.4140e-04 - val_loss: 0.0012
Epoch 6/8
463/463 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8.6302e-04 - val_loss: 0.0011
Epoch 7/8
463/463 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8.4528e-04 - val_loss: 0.0012
Epoch 8/8
463/463 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8.4813e-04 - val_loss: 0.0011
623/623 ━━━━━━━━━━━━━━━━━━━━ 1s 777us/step
--- MLP Performance --- RMSE: 130.54, MAE: 81.46, R²: 0.9258


## 3. Deep Learning Baseline 2: Gated Recurrent Unit (GRU)
Single-layer GRU recurrent network.

In [7]:
model_gru = Sequential([
    GRU(64, input_shape=(X_tr.shape[1], X_tr.shape[2]), return_sequences=False),
    Dense(32, activation='relu'),
    Dense(1)
])
model_gru.compile(optimizer='adam', loss='mse')
model_gru.fit(X_tr, y_tr, epochs=10, batch_size=128, validation_split=0.15, verbose=1)

pred_gru_s = model_gru.predict(X_te)
pred_gru = np.clip(scaler_y.inverse_transform(pred_gru_s).flatten(), 0, None)

gru_rmse = np.sqrt(mean_squared_error(y_true, pred_gru))
gru_mae = mean_absolute_error(y_true, pred_gru)
gru_r2 = r2_score(y_true, pred_gru)
print(f'--- GRU Performance --- RMSE: {gru_rmse:.2f}, MAE: {gru_mae:.2f}, R²: {gru_r2:.4f}')

c:\Users\tejas\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
463/463 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - loss: 0.0028 - val_loss: 0.0027
Epoch 2/10
463/463 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.0014 - val_loss: 0.0020
Epoch 3/10
463/463 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.0011 - val_loss: 0.0018
Epoch 4/10
463/463 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.0010 - val_loss: 0.0016
Epoch 5/10
463/463 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 9.1288e-04 - val_loss: 0.0015
Epoch 6/10
463/463 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 8.3870e-04 - val_loss: 0.0015
Epoch 7/10
463/463 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 7.8444e-04 - val_loss: 0.0014
Epoch 8/10
463/463 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 7.3787e-04 - val_loss: 0.0013
Epoch 9/10
463/463 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 7.0150e-04 - val_loss: 0.0013
Epoch 10/10
463/463 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 6.6966e-04 - val_loss: 0.0012
623/623 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
--- GRU Performance --- RMSE: 124.04, MAE: 68.67, R²: 0.9330


## 4. Deep Learning Model 3: Stacked Bidirectional LSTM (Tuned DL Winner)
Advanced architecture with forward/backward context processing.

In [8]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=True), input_shape=(X_tr.shape[1], X_tr.shape[2])),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])
model_bilstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')

callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]
history_lstm = model_bilstm.fit(X_tr, y_tr, epochs=20, batch_size=128, validation_split=0.15, callbacks=callbacks, verbose=1)

pred_lstm_s = model_bilstm.predict(X_te)
pred_lstm = np.clip(scaler_y.inverse_transform(pred_lstm_s).flatten(), 0, None)

lstm_rmse = np.sqrt(mean_squared_error(y_true, pred_lstm))
lstm_mae = mean_absolute_error(y_true, pred_lstm)
lstm_r2 = r2_score(y_true, pred_lstm)
print(f'--- Bi-LSTM Performance --- RMSE: {lstm_rmse:.2f}, MAE: {lstm_mae:.2f}, R²: {lstm_r2:.4f}')

Epoch 1/20


c:\Users\tejas\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


463/463 ━━━━━━━━━━━━━━━━━━━━ 14s 24ms/step - loss: 0.0033 - val_loss: 0.0020 - learning_rate: 0.0010
Epoch 2/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 15s 31ms/step - loss: 0.0015 - val_loss: 0.0017 - learning_rate: 0.0010
Epoch 3/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - loss: 0.0013 - val_loss: 0.0015 - learning_rate: 0.0010
Epoch 4/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - loss: 0.0011 - val_loss: 0.0015 - learning_rate: 0.0010
Epoch 5/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - loss: 0.0010 - val_loss: 0.0014 - learning_rate: 0.0010
Epoch 6/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - loss: 9.5955e-04 - val_loss: 0.0011 - learning_rate: 0.0010
Epoch 7/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - loss: 9.1229e-04 - val_loss: 0.0011 - learning_rate: 0.0010
Epoch 8/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - loss: 8.4685e-04 - val_loss: 0.0010 - learning_rate: 0.0010
Epoch 9/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - loss: 7.8920e-04 - val_loss: 9.7202e-04 - learn

## 5. Neural Architecture Benchmarking Summary

In [9]:
dl_comp = pd.DataFrame({
    'Neural Architecture': ['Dense MLP (No Recurrence)', 'GRU Network (Single Gate)', 'Stacked Bi-LSTM (Tuned)'],
    'RMSE': [mlp_rmse, gru_rmse, lstm_rmse],
    'MAE': [mlp_mae, gru_mae, lstm_mae],
    'R² Score': [mlp_r2, gru_r2, lstm_r2],
    'Notes': ['Fails sequence dependencies', 'Faster training, strong baseline', 'WINNER — Best sequence representation']
})
dl_comp

,Neural Architecture,RMSE,MAE,R² Score,Notes
0,Dense MLP (No Recurrence),130.544435,81.460138,0.925809,Fails sequence dependencies
1,GRU Network (Single Gate),124.037739,68.671447,0.933020,"Faster training, strong baseline"
2,Stacked Bi-LSTM (Tuned),113.464787,60.570101,0.943952,WINNER — Best sequence representation


## 6. Machine Learning vs Deep Learning Comparison (XGBoost vs Bi-LSTM)

In [10]:
xgb_path = 'data/predictions_xgboost.csv'
if os.path.exists(xgb_path):
    xgb_df = pd.read_csv(xgb_path)
    xgb_rmse = np.sqrt(mean_squared_error(xgb_df['Boarding_Count'], xgb_df['Pred_Tuned']))
    xgb_mae = mean_absolute_error(xgb_df['Boarding_Count'], xgb_df['Pred_Tuned'])
    xgb_r2 = r2_score(xgb_df['Boarding_Count'], xgb_df['Pred_Tuned'])
    
    peak_mask_seq = meta_te['Is_Peak_Hour'].values == 1
    lstm_peak_wmape = (np.abs(y_true[peak_mask_seq] - pred_lstm[peak_mask_seq]).sum() / y_true[peak_mask_seq].sum()) * 100
    
    xgb_peak = xgb_df[xgb_df['Is_Peak_Hour'] == 1]
    xgb_peak_wmape = (np.abs(xgb_peak['Boarding_Count'] - xgb_peak['Pred_Tuned']).sum() / xgb_peak['Boarding_Count'].sum()) * 100
    
    final_comp = pd.DataFrame({
        'Architecture': ['XGBoost (Tuned ML Winner)', 'Stacked Bi-LSTM (Tuned DL Winner)'],
        'RMSE': [xgb_rmse, lstm_rmse],
        'MAE': [xgb_mae, lstm_mae],
        'R² Score': [xgb_r2, lstm_r2],
        'Peak WMAPE (%)': [xgb_peak_wmape, lstm_peak_wmape]
    })
    display(final_comp)
else:
    print('XGBoost predictions file not found. Run Notebook 04 first.')

,Architecture,RMSE,MAE,R² Score,Peak WMAPE (%)
0,XGBoost (Tuned ML Winner),82.398314,43.852905,0.970102,8.766310
1,Stacked Bi-LSTM (Tuned DL Winner),113.464787,60.570101,0.943952,14.943078


## 7. Model Persistence

In [11]:
os.makedirs('models', exist_ok=True)
model_bilstm.save('models/lstm_tuned_model.keras')
print('Saved model: models/lstm_tuned_model.keras')

lstm_export = meta_te[['Date', 'Hour', 'Station', 'Boarding_Count', 'Is_Peak_Hour']].copy()
lstm_export['Pred_LSTM_Base'] = pred_gru
lstm_export['Pred_LSTM_Tuned'] = pred_lstm
lstm_export.to_csv('data/predictions_lstm.csv', index=False)
print(f'Exported LSTM test predictions ({lstm_export.shape[0]} rows) to data/predictions_lstm.csv')

Saved model: models/lstm_tuned_model.keras
Exported LSTM test predictions (19920 rows) to data/predictions_lstm.csv
